# Module 06 Lab - Regression and Classification Models**Objective:** To understand the difference between regression and classification and to build your first linear models for both tasks.**In this lab, you will write the code to train and evaluate the models.**

## Part 1: Regression vs. ClassificationThis is the most fundamental distinction between types of supervised learning problems.*   **Regression:** The goal is to predict a **continuous numerical value**.     *   *Examples:* Predicting the price of a house, the temperature tomorrow, or the stock price.*   **Classification:** The goal is to predict a **discrete category or class label**.    *   *Examples:* Predicting if an email is spam or not spam, if a flower is a setosa, versicolor, or virginica, or if a customer will churn or not.**In this lab, we will tackle one of each.**

## Part 2: Linear Regression**Concept:** Linear Regression is used to predict a continuous value. It works by finding the best-fitting straight line through the data points. The model learns a "slope" (coefficient) for each feature and an "intercept".*   **Problem:** We will predict the `Fare` of a Titanic passenger based on their `Age` and `Pclass`.

In [46]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.linear_model import LinearRegression, BayesianRidge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# **🛠️ Phase: Data Data Preprocessing and Multi-Stage Train-Validation-Test Splitting**

In [47]:
# 1. Load Data
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

# Drop row with missing Age (Clean missing values)
df.dropna(subset=['Age', 'Fare'], inplace=True)

# 2. ENCODE STRINGS TO NUMBERS (Converts 'male' to 0 and 'female' to 1)
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

# 3. Define features and target (Regression: Fare)
X = df[['Age', 'Pclass', 'Sex']]
y = df['Fare']

# 4. Data split performence

# Step 1: 70% Train, 30% Temporary
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)

# Step 2: Split Temporary into 15% Validation and 15% Test
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

# 5. Scale data (Essential for KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# **🛠️ Phase: Linear Regression Training and Evaluation**

### Task 1: Train and Evaluate a Linear Regression Model**Your Task:**1.  Create an instance of the `LinearRegression` model.2.  Train the model using the training data (`X_train_reg`, `y_train_reg`).3.  Make predictions on the test data.4.  Evaluate the model using `mean_squared_error`. This metric tells us the average of the squared differences between the predicted and actual values.

In [48]:
# 1. Create the model instance
lr_model = LinearRegression()
# 2. Train the model
lr_model.fit(X_train_scaled, y_train)
# 3. Make predictions
y_pred_reg = lr_model.predict(X_test_scaled)
# 4. Evaluate the model
mse = mean_squared_error(y_test, y_pred_reg)
print(f"Mean Squared Error for Fare Prediction: {mse:.2f}")
print(f"The square root of this is {np.sqrt(mse):.2f}, meaning our model is off by about ${np.sqrt(mse):.2f} on average.")

Mean Squared Error for Fare Prediction: 860.89
The square root of this is 29.34, meaning our model is off by about $29.34 on average.


# **🛠️ Phase: Comparative Analysis, Individual Models vs Ensemble Methods**

In [57]:
# --- ENSEMBLE AND EVALUATION BLOCK ---

# 1. Define the 5 required models
models_reg = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(),
    "Gradient Boosting": GradientBoostingRegressor(),
    "KNN": KNeighborsRegressor()
}

# 2. Train and evaluate individual models
reg_results = []
for name, model in models_reg.items():
    # Use variables created in split: y_train, y_val, y_test
    model.fit(X_train_scaled, y_train)
    val_preds = model.predict(X_val_scaled)
    test_preds = model.predict(X_test_scaled)

    reg_results.append({
        "Model": name,
        "Val MAE": mean_absolute_error(y_val, val_preds),
        "Val MSE": mean_squared_error(y_val, val_preds),
        "Val R2": r2_score(y_val, val_preds),
        "Test MAE": mean_absolute_error(y_test, test_preds),
        "Test MSE": mean_squared_error(y_test, test_preds), #
        "Test R2": r2_score(y_test, test_preds) #
    })

# 3. Ensemble: Voting Regressor (Best 3)
voting_reg = VotingRegressor(estimators=[
    ('rf', models_reg['Random Forest']),
    ('gb', models_reg['Gradient Boosting']),
    ('knn', models_reg['KNN'])
])
voting_reg.fit(X_train_scaled, y_train)

# 4. Ensemble: Bayesian Model
bayesian_model = BayesianRidge()
bayesian_model.fit(X_train_scaled, y_train)

# 5. Ensembles to results table
for name, ens_model in [("Voting Regressor", voting_reg), ("Bayesian Ensemble", bayesian_model)]:
    v_preds = ens_model.predict(X_val_scaled)
    t_preds = ens_model.predict(X_test_scaled)
    reg_results.append({
        "Model": name,
        "Val MAE": mean_absolute_error(y_val, v_preds),
        "Test MAE": mean_absolute_error(y_test, t_preds),
        "Test MSE": mean_squared_error(y_test, t_preds),
        "Test R2": r2_score(y_test, t_preds)
    })

# 6. Create and Display Table for PDF Deliverable
# Highlights the best R2
comparison_table = pd.DataFrame(reg_results)

styled_table = comparison_table.style.format({
    "Val MAE": "{:.2f}",
    "Val MSE": "{:.2f}",
    "Val R2": "{:.4f}",
    "Test MAE": "{:.2f}",
    "Test MSE": "{:.2f}",
    "Test R2": "{:.4f}"
}).background_gradient(cmap='Blues', subset=['Test R2'])

styled_table

,Model,Val MAE,Val MSE,Val R2,Test MAE,Test MSE,Test R2
0,Linear Regression,0.32,0.17,0.2996,0.30,0.15,0.3673
1,Decision Tree,0.22,0.16,0.3241,0.26,0.21,0.1590
2,Random Forest,0.24,0.14,0.4033,0.27,0.18,0.2624
3,Gradient Boosting,0.28,0.15,0.3939,0.29,0.16,0.3486
4,KNN,0.26,0.16,0.3204,0.27,0.17,0.2926
5,Voting Regressor,0.26,nan,nan,0.28,0.16,0.3443
6,Bayesian Ensemble,0.32,nan,nan,0.30,0.15,0.3679


# **🛠️ Phase: Implementing the Voting Regressor**

In [50]:
# Step 1: Create the Voting Regressor using the best 3 individual models (We use the models already initialized in models_reg dictionary)
voting_reg = VotingRegressor(estimators=[
    ('rf', models_reg['Random Forest']),
    ('gb', models_reg['Gradient Boosting']),
    ('knn', models_reg['KNN'])
])

# Step 2: Train the Voting Regressor
voting_reg.fit(X_train_scaled, y_train)

VotingRegressor(estimators=[('rf', RandomForestRegressor()),
                            ('gb', GradientBoostingRegressor()),
                            ('knn', KNeighborsRegressor())])

## Part 3: Logistic Regression**Concept:** Despite its name, Logistic Regression is a **classification** algorithm. It works by calculating the probability that a given input belongs to a certain class. It's one of the most widely used and interpretable classification models.*   **Problem:** We will predict whether a passenger `Survived` based on their `Age`, `Pclass`, and `Sex`.

# **🛠️ Phase: Data Data Preprocessing and Multi-Stage Train-Validation-Test Splitting**

In [51]:
# Load and Preprocess
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
df.dropna(subset=['Age'], inplace=True)
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

X = df[['Age', 'Pclass', 'Sex']]
y = df['Survived']

# Step 1: Split into 70% Train and 30% Temporary
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)

# Step 2: Split the 30% into half (15% Validation, 15% Test)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

# Scaling (Crucial for KNN and SVC)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

### Task 2: Train and Evaluate a Logistic Regression Model**Your Task:**1.  Create an instance of the `LogisticRegression` model.2.  Train the model using the classification training data.3.  Make predictions on the test data.4.  Evaluate the model using `accuracy_score`.

# **🛠️ Phase: Data Cleaning and Multi-Stage Dataset Splitting**

In [52]:
# 1. CLEANING: Drop any row where features or target are missing ( Age is the most common column with missing values the Titanic set)
features_cls = ['Age', 'Pclass', 'Sex']
df_clean = df.dropna(subset=features_cls + ['Survived'])

# 2. DEFINE FEATURES AND TARGET
X_cls = df_clean[features_cls]
y_cls = df_clean['Survived']

# 3. SPLIT: 70% Train, 30% Temporary (as per project guidelines)
X_train_cls, X_temp_cls, y_train_cls, y_temp_cls = train_test_split(
    X_cls, y_cls, test_size=0.30, random_state=42
)

# 4. SPLIT: Divide the 30% Temp into 15% Validation and 15% Test
X_val_cls, X_test_cls, y_val_cls, y_test_cls = train_test_split(
    X_temp_cls, y_temp_cls, test_size=0.50, random_state=42
)

# 5. MODEL: Create and Train
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_cls, y_train_cls)

# 6. EVALUATE: Predict and check accuracy
y_pred_cls = log_model.predict(X_test_cls)
accuracy = accuracy_score(y_test_cls, y_pred_cls)

print(f"Accuracy for Survival Prediction: {accuracy:.2%}")

Accuracy for Survival Prediction: 77.78%


# **🛠️ Phase: Comparative Evaluation of Classification Algorithms**

In [53]:
models = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "KNN": KNeighborsClassifier(),
    "SVC": SVC(probability=True)
}

results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_val_scaled)

    # Calculate Metrics
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, preds),
        "F1-Score": f1_score(y_val, preds),
        "ROC-AUC": roc_auc_score(y_val, model.predict_proba(X_val_scaled)[:, 1])
    })

val_results_df = pd.DataFrame(results)
print(val_results_df)

                 Model  Accuracy  F1-Score   ROC-AUC
0  Logistic Regression  0.757009  0.675000  0.807413
1        Decision Tree  0.803738  0.734177  0.834484
2        Random Forest  0.813084  0.756098  0.844113
3    Gradient Boosting  0.785047  0.729412  0.854833
4                  KNN  0.775701  0.692308  0.820131
5                  SVC  0.757009  0.638889  0.815407


# **🛠️ Phase: Implementing the Voting Classifier**

In [54]:
# Voting Ensemble (using top 3 models, e.g., RF, GB, and SVC)
voting_clf = VotingClassifier(estimators=[
    ('rf', models['Random Forest']),
    ('gb', models['Gradient Boosting']),
    ('svc', models['SVC'])
], voting='soft')

voting_clf.fit(X_train_scaled, y_train)

VotingClassifier(estimators=[('rf', RandomForestClassifier()),
                             ('gb', GradientBoostingClassifier()),
                             ('svc', SVC(probability=True))],
                 voting='soft')

# **🛠️ Phase: Comparative Analysis, Individual Models vs Ensemble Methods**

In [55]:
final_metrics = []

# 1. Loop through individual models
for name, model in models.items():
    val_preds = model.predict(X_val_scaled)
    test_preds = model.predict(X_test_scaled)

    final_metrics.append({
        "Model": name,
        "Val Accuracy": accuracy_score(y_val, val_preds),
        "Test Accuracy": accuracy_score(y_test, test_preds),
        "Test Precision": precision_score(y_test, test_preds),
        "Test Recall": recall_score(y_test, test_preds),
        "Test F1-Score": f1_score(y_test, test_preds)
    })

# 2. Add the Voting Ensemble to the list
ensemble_val_preds = voting_clf.predict(X_val_scaled)
ensemble_test_preds = voting_clf.predict(X_test_scaled)

final_metrics.append({
    "Model": "Voting Ensemble (Best 3)",
    "Val Accuracy": accuracy_score(y_val, ensemble_val_preds),
    "Test Accuracy": accuracy_score(y_test, ensemble_test_preds),
    "Test Precision": precision_score(y_test, ensemble_test_preds),
    "Test Recall": recall_score(y_test, ensemble_test_preds),
    "Test F1-Score": f1_score(y_test, ensemble_test_preds)
})

# 3. Create the DataFrame
comparison_table = pd.DataFrame(final_metrics)
comparison_table.to_csv("model_comparison.csv") # save this as PDF

# 4. Format for display (converts 0.85 to 85.0%)
styled_table = comparison_table.style.format({
    "Val Accuracy": "{:.2%}",
    "Test Accuracy": "{:.2%}",
    "Test Precision": "{:.2f}",
    "Test Recall": "{:.2f}",
    "Test F1-Score": "{:.2f}"
})

# SHOWS THE TABLE
styled_table

,Model,Val Accuracy,Test Accuracy,Test Precision,Test Recall,Test F1-Score
0,Logistic Regression,75.70%,77.78%,0.72,0.78,0.75
1,Decision Tree,80.37%,75.93%,0.75,0.65,0.70
2,Random Forest,81.31%,75.93%,0.72,0.72,0.72
3,Gradient Boosting,78.50%,76.85%,0.74,0.70,0.72
4,KNN,77.57%,75.00%,0.70,0.72,0.71
5,SVC,75.70%,76.85%,0.76,0.67,0.71
6,Voting Ensemble (Best 3),80.37%,76.85%,0.77,0.65,0.71


## 📝 Knowledge Check**Instructions:** Answer the following questions in this markdown cell.1.  **In your own words, what is the key difference between a regression problem and a classification problem?**2.  **The `LinearRegression` model has an attribute called `.coef_`. After you train the model, print `lr_model.coef_`. What do these numbers represent?**3.  **Why did we use `mean_squared_error` to evaluate the regression model but `accuracy_score` for the classification model?** Why wouldn't accuracy be a good metric for the fare prediction task?**[

1. Regression is for predict numbers like price or fare. Classification is for predict group or yes/no answer. Regression give many values, classification give only category.

2. The .coef_ numbers show how much each feature change the prediction. Bigger number mean more effect, smaller number mean less effect. It help to understand which feature is important for model.

3. We use mean squared error for regression because the number can be many different. Accuracy is for classification because it only right or wrong. Accuracy not good for fare because prediction not exactly same number.]**